In [2]:
import sys
import functools
import threading

def debug_flow(func):

    thread_local = threading.local()

    def tracer(frame, event, arg):

        if not hasattr(thread_local, "last_locals"):
            thread_local.last_locals = {}

        func_name = frame.f_code.co_name

        if event == "call":

            print(f"\nCALL → {func_name}")
            if frame.f_locals:
                print("ARGS:", frame.f_locals)

        elif event == "line":

            current = frame.f_locals
            previous = thread_local.last_locals

            for key, value in current.items():

                if key not in previous:
                    print(f"CREATE {func_name}.{key} = {value}")

                elif previous[key] != value:
                    print(
                        f"UPDATE {func_name}.{key}: "
                        f"{previous[key]} → {value}"
                    )

            thread_local.last_locals = current.copy()

        elif event == "return":

            print(f"RETURN ← {func_name} → {arg}")

        return tracer

    @functools.wraps(func)
    def wrapper(*args, **kwargs):

        sys.settrace(tracer)

        try:
            return func(*args, **kwargs)
        finally:
            sys.settrace(None)

    return wrapper

In [3]:
@debug_flow
def calculate(x):

    a = x + 10
    b = helper(a)

    result = b * 2

    return result


def helper(v):

    temp = v + 5
    return temp

In [4]:
calculate(5)


CALL → calculate
ARGS: {'x': 5}
CREATE calculate.x = 5
CREATE calculate.a = 15

CALL → helper
ARGS: {'v': 15}
CREATE helper.v = 15
CREATE helper.temp = 20
RETURN ← helper → 20
CREATE calculate.x = 5
CREATE calculate.a = 15
CREATE calculate.b = 20
CREATE calculate.result = 40
RETURN ← calculate → 40


40

In [5]:
def debug_tree(func):

    state = threading.local()

    def tracer(frame, event, arg):

        if not hasattr(state, "depth"):
            state.depth = 0

        if not hasattr(state, "locals_stack"):
            state.locals_stack = {}

        func_name = frame.f_code.co_name
        indent = "│   " * state.depth

        if event == "call":

            print(f"{indent}├── CALL {func_name}")
            if frame.f_locals:
                print(f"{indent}│   ARGS: {frame.f_locals}")

            state.locals_stack[frame] = {}
            state.depth += 1


        elif event == "line":

            current = frame.f_locals
            previous = state.locals_stack.get(frame, {})

            for key, value in current.items():

                if key not in previous:
                    print(f"{indent}│   CREATE {key} = {value}")

                elif previous[key] != value:
                    print(
                        f"{indent}│   UPDATE {key}: "
                        f"{previous[key]} → {value}"
                    )

            state.locals_stack[frame] = current.copy()


        elif event == "return":

            state.depth -= 1
            indent = "│   " * state.depth

            print(f"{indent}└── RETURN {func_name} → {arg}")

        return tracer


    @functools.wraps(func)
    def wrapper(*args, **kwargs):

        sys.settrace(tracer)

        try:
            return func(*args, **kwargs)
        finally:
            sys.settrace(None)

    return wrapper

In [6]:
def helper(x):
    temp = x + 5
    return temp


def compute(v):
    a = v * 2
    b = helper(a)
    return b

@debug_tree
def main_api(x):

    result = compute(x)

    final = result + 1
    return final

In [7]:
main_api(5)

├── CALL main_api
│   ARGS: {'x': 5}
│   │   CREATE x = 5
│   ├── CALL compute
│   │   ARGS: {'v': 5}
│   │   │   CREATE v = 5
│   │   │   CREATE a = 10
│   │   ├── CALL helper
│   │   │   ARGS: {'x': 10}
│   │   │   │   CREATE x = 10
│   │   │   │   CREATE temp = 15
│   │   └── RETURN helper → 15
│   │   │   CREATE b = 15
│   └── RETURN compute → 15
│   │   CREATE result = 15
│   │   CREATE final = 16
└── RETURN main_api → 16


16

In [ ]:
import time
import inspect


def debug_runtime(func):

    state = threading.local()

    def init_state():
        if not hasattr(state, "depth"):
            state.depth = 0
            state.stack = []
            state.locals_map = {}
            state.tree = {"name": func.__name__, "children": []}
            state.node_stack = [state.tree]

    def tracer(frame, event, arg):

        init_state()

        func_name = frame.f_code.co_name
        indent = "│   " * state.depth

        if event == "call":

            node = {
                "name": func_name,
                "start": time.time(),
                "children": [],
                "vars": {}
            }

            parent = state.node_stack[-1]
            parent["children"].append(node)

            state.node_stack.append(node)
            state.locals_map[frame] = {}
            state.depth += 1

            print(f"{indent}├── CALL {func_name}")

        elif event == "line":

            current = frame.f_locals
            previous = state.locals_map.get(frame, {})

            for k, v in current.items():

                if k not in previous:
                    print(f"{indent}│   CREATE {k} = {v}")

                elif previous[k] != v:
                    print(
                        f"{indent}│   UPDATE {k}: "
                        f"{previous[k]} → {v}"
                    )

            state.locals_map[frame] = current.copy()

        elif event == "return":

            state.depth -= 1
            indent = "│   " * state.depth

            node = state.node_stack.pop()
            node["return"] = arg
            node["duration"] = round(time.time() - node["start"], 6)

            print(
                f"{indent}└── RETURN {func_name} "
                f"→ {arg} ({node['duration']}s)"
            )

        return tracer

    async def async_wrapper(*args, **kwargs):

        sys.settrace(tracer)

        try:
            result = await func(*args, **kwargs)
        finally:
            sys.settrace(None)

        print("\nExecution Tree:")
        print_tree(state.tree)

        return result

    def sync_wrapper(*args, **kwargs):

        sys.settrace(tracer)

        try:
            result = func(*args, **kwargs)
        finally:
            sys.settrace(None)

        # print("\nExecution Tree:")
        # print_tree(state.tree)

        return result

    if inspect.iscoroutinefunction(func):
        return functools.wraps(func)(async_wrapper)

    return functools.wraps(func)(sync_wrapper)
 

def print_tree(node, indent=0):

    space = "  " * indent

    if "duration" in node:
        print(
            f"{space}- {node['name']} "
            f"(time={node['duration']}s)"
        )
    else:
        print(f"{space}- {node['name']}")

    for child in node.get("children", []):
        print_tree(child, indent + 1)

In [18]:
def helper(x):
    temp = x + 5
    return temp

def compute(v):
    a = v * 2
    b = helper(a)
    return b
 
@debug_runtime
def main_api(x):

    result = compute(x)

    final = result + 1
    return final

In [19]:
main_api(10)

├── CALL main_api
│   │   CREATE x = 10
│   ├── CALL compute
│   │   │   CREATE v = 10
│   │   │   CREATE a = 20
│   │   ├── CALL helper
│   │   │   │   CREATE x = 20
│   │   │   │   CREATE temp = 25
│   │   └── RETURN helper → 25 (3.1e-05s)
│   │   │   CREATE b = 25
│   └── RETURN compute → 25 (9.6e-05s)
│   │   CREATE result = 25
│   │   CREATE final = 26
└── RETURN main_api → 26 (0.000208s)


26

In [23]:
import sys
import functools
import threading
import inspect
import time


def debug_runtime(func):

    state = threading.local()

    def init_state():
        if not hasattr(state, "depth"):
            state.depth = 0
            state.locals_map = {}

    def format_call(frame):

        func_name = frame.f_code.co_name
        args = frame.f_locals

        arg_parts = []

        for name, value in args.items():

            if value is None:
                arg_parts.append(f"{name}=None")

            elif value == "" or value == [] or value == {}:
                arg_parts.append(f"{name}=EMPTY")

            else:
                arg_parts.append(f"{name}={value!r}")

        return f"{func_name}({', '.join(arg_parts)})"


    def tracer(frame, event, arg):

        init_state()

        indent = "│   " * state.depth
        func_name = frame.f_code.co_name

        if event == "call":

            call_string = format_call(frame)

            print(f"{indent}├── CALL {call_string}")

            state.locals_map[frame] = {}
            state.depth += 1


        elif event == "line":

            current = frame.f_locals
            previous = state.locals_map.get(frame, {})

            for k, v in current.items():

                if k not in previous:
                    print(f"{indent}│   CREATE {k} = {v!r}")

                elif previous[k] != v:
                    print(
                        f"{indent}│   UPDATE {k}: "
                        f"{previous[k]!r} → {v!r}"
                    )

            state.locals_map[frame] = current.copy()


        elif event == "return":

            state.depth -= 1
            indent = "│   " * state.depth

            print(
                f"{indent}└── RETURN {func_name} → {arg!r}"
            )

        return tracer


    @functools.wraps(func)
    def wrapper(*args, **kwargs):

        sys.settrace(tracer)

        try:
            return func(*args, **kwargs)
        finally:
            sys.settrace(None)

    return wrapper

In [24]:
def helper(a, b=None):
    total = a + (b or 0)
    return total


def compute(x, y=None):
    value = helper(x, y)
    return value


@debug_runtime
def main_api(user_id, data=None):

    result = compute(user_id, data)

    final = result * 2
    return final

In [25]:
main_api(5, None)

├── CALL main_api(user_id=5, data=None)
│   │   CREATE user_id = 5
│   │   CREATE data = None
│   ├── CALL compute(x=5, y=None)
│   │   │   CREATE x = 5
│   │   │   CREATE y = None
│   │   ├── CALL helper(a=5, b=None)
│   │   │   │   CREATE a = 5
│   │   │   │   CREATE b = None
│   │   │   │   CREATE total = 5
│   │   └── RETURN helper → 5
│   │   │   CREATE value = 5
│   └── RETURN compute → 5
│   │   CREATE result = 5
│   │   CREATE final = 10
└── RETURN main_api → 10


10